# Job 49744985 — the layer-0 features were free (finding 21)

**Question.** Job 49729375 (finding 19) pre-registered that the leak-free model would have
*no* `next_node`-family SAE features at layer 0 while the leaked record model still would.
The counts came back **23 vs 24** — essentially identical — and I recorded that pre-registration
as FALSIFIED while noting it was probably *untestable as written*. This job settles which.

**Why it may have been untestable.** Reasoning positions come in two kinds and the position index
means a different thing in each (`spin_gnn/sudoku_activations.py:36-75`):

| kind | offset | `tokens[pos]` is | consequence |
|---|---|---|---|
| `pre_node` | `pos = i - 1` | `BEGIN_ROUND` or the previous deduction's colour | nothing at the embedding identifies the node that comes next |
| `pre_color` | `pos = i` | **the node itself**, whose colour is the target | `next_node[v]` is matched *for free* by any token-identity feature, at layer 0, in both arms, with no leak |

Pooling the two kinds, as 49729375 did, mixes a free match into the count.

**What is new.** `--split-by-kind` re-scores each already-fitted dictionary twice more — once on
`pre_node` positions only, once on `pre_color` only — into `sae_summary_by_kind.csv`. The SAE fits,
the predicate set, the aliveness floor and the rolled null are unchanged, and `position_mask=None`
is the code path 49729375 ran, untouched, so **`sae_summary.csv` must reproduce exactly**.

Pre-registrations (a)–(e) are in the header of `scripts/slurm/sudoku2_sae_kindsplit.sbatch`,
written before the job ran.

## Provenance

| | |
|---|---|
| Slurm job | **49744985** (`sud2saeKS`), COMPLETED 01:00:56, ExitCode 0:0 |
| script | `scripts/slurm/sudoku2_sae_kindsplit.sbatch` |
| code commit | `f24bb40` (`position_mask` + `--split-by-kind` + 2 regression tests) |
| notebook commit | `1360c70` (`STATUS: point next step 1 at the prepared L*(T) ladder script`) |
| SAE seed | `--seed 0` |
| SAE config | `--expansion 16 --l1-coeff 0.1 --l1-max-tries 8 --sae-epochs 1500 --n-instances 3000 --max-positions-per-instance 40 --alive-rate 1e-3 --n-null-shifts 19 --target-l0 64` |
| layers | 0, 2, 4, 6, 8, 10, 12 |
| leak-free checkpoint | `$SCRATCH/spin_gnn/sudoku_n24_leakfree_2026-10-26/d0256_h04_L12_ff01024_lr1e-03_pre_shuf` (shared-pool solve_rate 0.8540) |
| leaked checkpoint | `$OAK/hshunt/checkpoints/spin_gnn/sudoku_dd_d0256_L12_ntrain80000_ep300_sharedpool0.9960_job48912548` (0.9960) |
| instance config | N=24, degree=5, q=4, rho=0.3, rho_max=0.95, removal hints; leak-free arm `shuffle_adj=True` |
| previous run on the same eval | job **49729375**, `$SCRATCH/spin_gnn/sudoku_sae_2026-10-26_leakfree_l1fix/{leakfree,leaked}_l0_64` |

In [ ]:
import ast, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SCRATCH = os.environ["SCRATCH"]
RUN  = f"{SCRATCH}/spin_gnn/sudoku_sae_kindsplit_2026-10-27"          # job 49744985
PREV = f"{SCRATCH}/spin_gnn/sudoku_sae_2026-10-26_leakfree_l1fix"      # job 49729375
LAYERS = [0, 2, 4, 6, 8, 10, 12]
FAMILIES = ("next_node", "frontier", "justifies_next", "nbr_of_next")
BLUE, RED = "#1f77b4", "#d62728"

def by_kind(arm):
    df = pd.read_csv(f"{RUN}/{arm}_l0_64/sae_summary_by_kind.csv")
    fam = df.family_counts.map(ast.literal_eval)
    for f in FAMILIES:
        df[f"n_{f}"] = fam.map(lambda d, f=f: d.get(f, 0))
    return df.set_index(["layer", "position_kind"]).sort_index()

lf, lk = by_kind("leakfree"), by_kind("leaked")
print(lf.columns.tolist())

## 1. Reproduction check — the pooled result must be unchanged

This is the comparison to the previous best on the same eval. The metric of record for an SAE pass
is `sae_summary.csv`, and because `position_mask=None` is exactly the path 49729375 ran, the
requirement is not "similar" but **identical**.

In [ ]:
for arm in ("leakfree", "leaked"):
    new = pd.read_csv(f"{RUN}/{arm}_l0_64/sae_summary.csv")
    old = pd.read_csv(f"{PREV}/{arm}_l0_64/sae_summary.csv")
    # `checkpoint` is an absolute path and is identical by construction; compare the rest.
    cols = [c for c in new.columns if c != "checkpoint"]
    print(f"{arm:9s} equals={new[cols].equals(old[cols])}  rows={len(new)}")

Both arms return `True`: **finding 19 reproduces byte-for-byte**, and the `position_mask=None`
path is provably unchanged by this commit. Everything below is additive.

## 2. The metrics table — layer 0, decomposed

The headline. 49729375's pooled 23-vs-24 count, split by position kind.

In [ ]:
rows = []
for arm, df in (("leak-free (0.854)", lf), ("leaked record (0.996)", lk)):
    for kind in ("pre_node", "pre_color"):
        r = df.loc[(0, kind)]
        rows.append(dict(model=arm, positions=kind, n=int(r.n_positions),
                         alive=int(r.n_alive), interpretable=int(r.n_interpretable),
                         next_node_feats=int(r.n_next_node),
                         best_next_node_f1=round(r.best_next_node_f1, 3),
                         median_null_f1=round(r.median_best_f1_null, 3)))
layer0 = pd.DataFrame(rows)
layer0

**Verdict on (a), (b), (c).**

- **(a) CONFIRMED.** Leak-free, layer 0, `pre_node`: **0** `next_node`-family features,
  `best_next_node_f1` = **0.000**, below its own median null (0.092). Nothing ordered leaks through
  the permuted encoding at the embedding.
- **(b) CONFIRMED.** Leak-free, layer 0, `pre_color`: **23** features at F1 **exactly 1.000** — the
  signature of a pure token-identity feature. All 23 of 49729375's layer-0 features were these.
- **(c) CONFIRMED, in the branch I predicted.** The leaked arm decomposes *identically*
  (0 at `pre_node` with F1 0.011; 24 at `pre_color` at F1 1.000). So the 23-vs-24 count compared two
  freebies and could not have separated the arms under any outcome.
  **49729375's pre-registration (a) is reclassified from FALSIFIED to UNTESTABLE.**

Layer 0 is the token+position embedding and the only layer in either arm that misses the sparsity
target (realized L0 5.9 / 7.0 against 64): a 2-sparse embedding is what the dictionary finds because
that is what is there.

## 3. Where the DD families actually live, layer by layer

Pre-registration (d) said at least one `justifies_next` feature would survive at `pre_node`
positions in the leak-free arm at L8 or L12, and that the leak-free arm would beat the leaked one
there. The falsifier I wrote down was: *all* of finding 19's edge features sit at `pre_color` only.

In [ ]:
def series(df, kind, col):
    return [df.loc[(L, kind), col] for L in LAYERS]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
panels = [("best_justifier_f1", "justifies_next[u]  (the edges)"),
          ("best_frontier_f1",  "frontier[v]  (selects the node)"),
          ("best_next_node_f1", "next_node[v]"),
          ("best_nbr_of_next_f1", "nbr_of_next[u]")]
for ax, (col, title) in zip(axes.ravel(), panels):
    for df, arm, ls, lw, a in ((lf, "leak-free", "-", 1.8, 1.0), (lk, "leaked", "--", 1.1, 0.55)):
        for kind, c in (("pre_node", BLUE), ("pre_color", RED)):
            ax.plot(LAYERS, series(df, kind, col), ls, color=c, marker="o", ms=4,
                    lw=lw, alpha=a, label=f"{arm} {kind}")
    ax.axhline(0.3, color="0.6", ls=":", lw=1.0)   # the interpretability cut
    ax.set_title(title); ax.set_xlabel("layer"); ax.set_ylabel("best F1")
    ax.set_ylim(0, 1.05); ax.set_xticks(LAYERS)
axes[0, 0].legend(fontsize=8, loc="upper left")
fig.suptitle("best F1 per DD family, by position kind (dotted = F1>=0.3 cut)")
fig.tight_layout()

In [ ]:
# Feature counts per family, summed over the layers above the embedding.
above0 = [L for L in LAYERS if L > 0]
counts = pd.DataFrame(
    {(arm, kind): {f: sum(int(df.loc[(L, kind), f"n_{f}"]) for L in above0) for f in FAMILIES}
     for arm, df in (("leak-free", lf), ("leaked", lk))
     for kind in ("pre_node", "pre_color")})
counts

In [ ]:
# The two numbers pre-reg (d) turns on, stated per layer rather than pooled.
jn = pd.DataFrame({
    "lf pre_node F1":  series(lf, "pre_node",  "best_justifier_f1"),
    "lk pre_node F1":  series(lk, "pre_node",  "best_justifier_f1"),
    "lf pre_color F1": series(lf, "pre_color", "best_justifier_f1"),
    "lk pre_color F1": series(lk, "pre_color", "best_justifier_f1"),
    "lf pre_color n":  [int(x) for x in series(lf, "pre_color", "n_justifies_next")],
    "lk pre_color n":  [int(x) for x in series(lk, "pre_color", "n_justifies_next")],
}, index=LAYERS).round(3)
jn.index.name = "layer"
jn

**Verdict on (d): FALSIFIED, on both clauses.**

- **Zero** `justifies_next` features at `pre_node` at *every* layer in *both* arms (best F1
  0.171–0.192 leak-free, 0.233–0.254 leaked — all under the 0.3 cut).
- And leak-free is *lower* there, not higher (0.171 vs 0.241 at L12), which is the opposite of the
  second clause.

All of finding 19's edge structure sits at **`pre_color`**: the leak-free arm has **69**
`justifies_next` features there (18 @ L8, 30 @ L10, 21 @ L12) against the leaked arm's **15**
(0 @ L8, 13 @ L10, 2 @ L12), best F1 **0.743 vs 0.247 at L8**.

So the leak-free vs leaked contrast survives the split and is in fact **larger inside the mask than
it was pooled** (0.486 vs 0.162 pooled → 0.743 vs 0.247 at `pre_color`). What shrinks is the claim:
from *"the model works out which neighbours constrain the node it is choosing"* to *"the model
represents the known neighbours while choosing that node's colour"*. Read positively, the second is
the time DD's firing rule actually needs — a node fires when its neighbours cover q−1 colours, and
the colour follows from exactly those neighbours. But it is the weaker of the two claims, because at
`pre_color` the node is already in the context window.

**Verdict on (e): SPLIT.** `best_frontier_f1` is higher at `pre_node` than at `pre_color` in 4 of 6
layers above the embedding (the pre-registered metric, so (e) is not carried on counts), but the
frontier *feature counts* favour `pre_node` at 6 of 6: 31/31/26/26/35/33 vs 13/14/19/15/14/2. The
frontier is the mirror image of the edges — it lives where the node is being selected, which is the
time its own definition needs.

## 4. New open problem found here: `pre_node` `next_node` features may be counting

Not pre-registered, and it points at a control that has never been run.

In [ ]:
nn = pd.DataFrame({
    "lf n":  [int(lf.loc[(L, "pre_node"), "n_next_node"]) for L in LAYERS],
    "lf F1": [round(lf.loc[(L, "pre_node"), "best_next_node_f1"], 3) for L in LAYERS],
    "lk n":  [int(lk.loc[(L, "pre_node"), "n_next_node"]) for L in LAYERS],
    "lk F1": [round(lk.loc[(L, "pre_node"), "best_next_node_f1"], 3) for L in LAYERS],
}, index=LAYERS)
nn.index.name = "layer"
nn

The **leaked** model has *more* `next_node` features at `pre_node` than the leak-free one
(60 vs 15 at L12, best F1 0.741 vs 0.545) — it is better at knowing which node comes next *before*
emitting it. That is the right direction for a model that solves the task by decoding token order,
but it also exposes a regularity neither arm controls for:

- Finding 16 measured that round events are emitted in **ascending node index for 100% of rounds**.
- `shuffle_adj` (`spin_gnn/config.py:184`) permutes each node's **neighbour list only** — verified
  by reading it. It does not touch the order in which round events are emitted.

So a `next_node` feature at `pre_node` may be tracking *"the index goes up"* rather than deduction,
**in both arms**. Permuting within-round event order is a separate, cheap control that has never
been run, and it is the next thing this analysis needs.

## Caveats that apply to every number above

- Each mask keeps 26,600 of 53,200 positions, so a feature at the 1e-3 aliveness floor fires ~26
  times instead of ~52. **Counts compare within a mask, not across masks.**
- The dictionaries are the pooled ones, scored twice. This does not ask what a dictionary *fitted*
  to `pre_node` positions alone would look like.
- One SAE seed, one checkpoint per arm. `best_f1` is still a max over 256 predicates; the rolled
  19-shift null controls the argmax, not the interpretation. A second seed is required before any of
  this becomes a paper figure.
- None of this is evidence that either model *runs* DD. It is evidence about which DD predicates are
  linearly readable from the residual stream, and at which positions.

## Slide deck

`notebooks/sudoku_sae_kindsplit_2026-10-27/sae_kindsplit.pdf` (8 pages) carries the same result for
reading away from a terminal, including the tokenization explainer and a real reasoning trace.